# BCI Competition IV-2a: oscillatory SSM with phase-synchrony readout (source-only screening)

Branch `feature/ssm-dan-osc-phase-sync`. Chronological leave-one-subject-out (9 subjects, 22 channels, 4 classes), no adaptation (no RA, no Stage 1, no IM-TTA).

The oscillatory branch (`OscillatorySSM`, `ssm_dan/models/backbone.py`) runs at 250 Hz: spatial filters give 16
virtual channels, each driving 8 damped oscillators with learned centre frequency and bandwidth (Hz). This branch adds
a **phase-synchrony readout** (`SyncProbes`): for every band k and probe j,
R_jk(t) = |sum_s w_jsk p_sk(t)| / sum_s |w_jsk|, with unit phasors p = h/|h| and learned complex weights w, so R in [0, 1]
measures how stable the phase relation between virtual channels is, independently of their amplitudes. Frequencies are
shared by all virtual channels when synchrony is read. Variants (applied in the listed order):

* `osc` / `osc_only`: the oscillatory branch feeds / replaces the token-rate selective SSM;
* `sync`: log band power + synchrony; `sync_only`: synchrony alone;
* `hemi`: half of the virtual channels see left + midline electrodes, half midline + right (standard_1020 montage);
  the probes start as left, right and global probes.

**Step 3b runs the unit tests and the synthetic learning check (`tests/test_osc_sync.py`) first and stops the
notebook if any fails**, before any real data are trained.

Runs on Kaggle (GPU T4 x2, **Internet on**). Every number printed or plotted is read from the runs' own output,
`results/<dataset>/<run>/seed<k>/`; runs of the earlier oscillatory notebooks enter the comparison table when their
output is attached (*Add Input -> Your Work -> Notebook Output*), which also resumes unfinished runs.

## 1. Settings

In [ ]:
DATASET = 'bcic2a'
# (method, variant) pairs to run; variants are applied in the listed order.
EXPERIMENTS = [
    ('ssm_dan', 'source_only,osc,sync,hemi'),          # power + synchrony, feeds the token SSM
    ('ssm_dan', 'source_only,osc_only,sync,hemi'),     # power + synchrony, replaces the token SSM
    # ('ssm_dan', 'source_only,osc_only,sync_only,hemi'),  # synchrony alone
    # ('ssm_dan', 'source_only'),                      # current backbone (about 6 h)
]
SUBJECTS = 'all'        # 'all' | '3' | '1-4' | '1,5,9'
SEED = 0
GPUS = [0, 1]           # Kaggle T4 x2; one GPU: [0]
MODE = 'paper'          # 'paper': folds launched as in the reported runs; 'per_fold' | 'one_process'
OVERRIDES = []          # e.g. ['max_epochs=2'] for a quick check
TIME_BUDGET_H = None    # None: 11.5 h on Kaggle/Colab, no limit elsewhere
BRANCH = 'feature/ssm-dan-osc-phase-sync'

## 2. Get the code

In [ ]:
import os, subprocess, sys
from pathlib import Path

REPO = Path('/tmp/ssm-dan')                 # outside the saved output
URLS = ['https://github.com/CuongDM1806/ssm-dan.git',
        'https://github.com/CuongDM1806/tcformer-test.git']   # former name of the repository
for url in URLS:
    if (REPO / '.git').exists():
        break
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', BRANCH, url, str(REPO)],
                   env={**os.environ, 'GIT_TERMINAL_PROMPT': '0'})
subprocess.run(['git', '-C', str(REPO), 'log', '-1', '--oneline'], check=True)
sys.path.insert(0, str(REPO))
from ssm_dan import launch, results
from ssm_dan import config as cfg
from ssm_dan.data import SUBJECT_IDS

platform = launch.detect()                  # kaggle | colab | local
RESULTS = platform.results
CONFIGS = {}
for method, variant in EXPERIMENTS:
    variants = [v.strip() for v in variant.split(',') if v.strip()]
    config = cfg.resolve(method, DATASET, variants, OVERRIDES, seed=SEED)
    mode = launch.paper_process_mode(DATASET, variants) if MODE == 'paper' else MODE
    CONFIGS[config['name']] = dict(method=method, variant=variant, mode=mode, config=config)
TARGETS = cfg.parse_subjects(SUBJECTS, SUBJECT_IDS[DATASET])
print('platform:', platform.name, '| results:', RESULTS)
for name, run in CONFIGS.items():
    print(f'  {name:32s} mode {run["mode"]}')

## 3. Python environment

In [ ]:
PYTHON = launch.python_for_training(          # creates a Python 3.10 environment once if needed
    platform, ['pytest'] + (['transformers==4.48.3'] if any(m == 'eegencoder' for m, _ in EXPERIMENTS) else []))
env = launch.environment(platform)
GPUS = GPUS[:max(launch.available_gpus() or 1, 1)]
print('python:', PYTHON, '| GPUs used:', GPUS)

## 3b. Tests (stop here if any fails)

`tests/test_osc_sync.py` in the training environment: backward compatibility with the branch without synchrony
(same parameters, state_dict and output; strict loading), every `osc_branch` x `osc_sync` combination forward and
backward without NaN (also for a zero input), R in [0, 1], R > 0.95 for a common 10-Hz sine, invariance to a sign flip
compensated by a phase of pi, R < 0.5 for drifting phases, and the synthetic learning check: two classes that differ
only in the phase relation between the electrodes of one side; synchrony must exceed 95 %, power with frozen
electrode filters must stay near chance (with learned spatial filters its accuracy is printed only).

In [ ]:
test = subprocess.run([str(PYTHON), '-m', 'pytest', '-q', '-rA', '-s', 'tests/test_osc_sync.py'],
                      cwd=REPO, env=env, capture_output=True, text=True)
print(test.stdout[-12000:], test.stderr[-3000:])
if test.returncode != 0:
    raise SystemExit('Tests failed: no training on real data. Fix the branch first.')

## 4. Data

BCI Competition IV-2a comes from the official competition files only (see `data/README.md`):
`BCICIV_2a_gdf.zip` (GDF recordings) and `ds2a/true_labels.zip` (labels of all sessions) from bbci.de.

1. If the MAT files that training reads are already in `data/` (`data/BCICIV_2a/mat/A01T.mat` ... `A09E.mat`), they are used as they are.
2. Otherwise they are built from `data/BCICIV_2a/gdf/` (unzipped GDF) and `data/BCICIV_2a/true_labels/`
   (unzipped labels), or from these files in the attached inputs: on Kaggle, *Add Input* a Dataset with
   `BCICIV_2a_gdf.zip` and the true-label zip (renamed e.g. `2a_true_labels.zip`), zipped or unzipped.
3. If they are not found, both zips are downloaded from bbci.de and extracted into those folders. `mat/` is never stored.

In [ ]:
launch.ensure_data(DATASET, platform=platform, env=env, python=PYTHON)

## 5. Train

Each GPU takes the next experiment from the list and trains all its pending folds in one process (as in the reported runs; `per_session_ra` uses one process per fold). Progress is printed with its GPU and written to `results/bcic2a/<run>/seed<k>/logs/`. Each
experiment gets the time left, so no fold is started that would not finish before the session ends.

In [ ]:
import queue, threading, time

budget = TIME_BUDGET_H if TIME_BUDGET_H is not None else (11.5 if platform.name in ('kaggle', 'colab') else None)
started = time.monotonic()
failed = {}

def hours_left():
    return None if budget is None else budget - (time.monotonic() - started) / 3600

def train(name, gpus):
    run, left = CONFIGS[name], hours_left()
    if left is not None and left < 0.5:
        print(f'{name} not started: {left:.1f} h left', flush=True)
        return
    try:
        launch.run(run['method'], DATASET, variant=run['variant'], subjects=SUBJECTS, seed=SEED, gpus=gpus,
                   overrides=OVERRIDES, mode=run['mode'], time_budget_h=left, platform=platform)
    except Exception as error:              # keep the other experiments going
        failed[name] = repr(error)
        print(f'{name} FAILED: {error!r}', flush=True)

# One-process experiments: one per GPU at a time.
todo = queue.Queue()
for name, run in CONFIGS.items():
    if run['mode'] == 'one_process':
        todo.put(name)

def gpu_worker(gpu):
    while True:
        try:
            name = todo.get_nowait()
        except queue.Empty:
            return
        train(name, [gpu])

threads = []
for gpu in GPUS:
    if todo.empty():
        break
    threads.append(threading.Thread(target=gpu_worker, args=(gpu,)))
    threads[-1].start()
    time.sleep(60 if len(GPUS) > 1 else 0)  # stagger the data loading of parallel runs
for thread in threads:
    thread.join()

# Per-fold experiments: one after the other, one fold per GPU at a time.
for name, run in CONFIGS.items():
    if run['mode'] == 'per_fold':
        train(name, GPUS)

print(f'Finished in {(time.monotonic() - started) / 3600:.2f} h.', f'Failed: {failed}' if failed else '')

## 6. Results of the runs

Read from each run's `subjects/SXX.json`: accuracy (%) on session E of every target subject; for SSM-DAN
also `no TTA`, the same weights before IM-TTA (the Stage-1 UDA model). Below: mean and population
SD over the finished subjects, mean Cohen's kappa, training time per fold and the number of finished folds.

In [ ]:
import numpy as np

def run_records(name):
    return {r['subject_id']: r for r in results.load_subjects(RESULTS / DATASET / name / f'seed{SEED}')}

RECORDS = {name: run_records(name) for name in CONFIGS}
columns = []                                  # (header, run, metric)
for name, records in RECORDS.items():
    columns.append((name, name, 'test_acc'))
    for metric, label in (('no_tta_acc', 'no TTA'), ('all_sessions_acc', 'all sessions')):
        if any(metric in r for r in records.values()):
            columns.append((f'{name} | {label}', name, metric))

def cell(value, width):
    return f'{value:>{width}}'

widths = [max(len(header), 8) for header, _, _ in columns]
print(cell('subject', 8), *[cell(h, w) for (h, _, _), w in zip(columns, widths)])
for subject in TARGETS:
    values = [RECORDS[run].get(subject, {}).get(metric) for _, run, metric in columns]
    print(cell(f'S{subject}', 8), *[cell('-' if v is None else f'{v * 100:.2f}', w) for v, w in zip(values, widths)])

def summary(label, function):
    out = []
    for (_, run, metric), w in zip(columns, widths):
        values = [r[metric] for s, r in RECORDS[run].items() if s in TARGETS and metric in r]
        out.append(cell(function(values, run, metric) if values else '-', w))
    print(cell(label, 8), *out)

summary('mean', lambda v, run, metric: f'{np.mean(v) * 100:.2f}')
summary('SD', lambda v, run, metric: f'{np.std(v) * 100:.2f}')
summary('kappa', lambda v, run, metric: f'{np.mean([RECORDS[run][s][metric.replace("acc", "kappa")] for s in RECORDS[run] if s in TARGETS]):.3f}')
summary('min/fold', lambda v, run, metric: f'{np.nanmean([r.get("train_time_min", np.nan) for r in RECORDS[run].values()]):.1f}')
summary('folds', lambda v, run, metric: f'{len(v)}/{len(TARGETS)}')

## 7. Comparison table

All BCI Competition IV-2a runs of seed `SEED` under `results/` (also those of earlier sessions), on the subjects every run
has finished: accuracy per subject, mean +- SD (population SD), kappa, ITR and the two-sided Wilcoxon
signed-rank test against `REFERENCE` with Holm's correction. The paper tests each target-data access group separately, e.g. `REFERENCE = 'ssm_dan-source_only'` with `ONLY = 'eegconformer,atcnet,tcformer,eegencoder,satransnet'`. Written to `results/tables/bcic2a.csv`.

In [ ]:
REFERENCE = 'ssm_dan'
ONLY = ''               # comma-separated runs; '' = every run
METRIC = 'test_acc'     # test_acc | no_tta_acc | all_sessions_acc

command = [str(PYTHON), '-m', 'ssm_dan.tables', '--results', str(RESULTS), '--dataset', DATASET,
           '--reference', REFERENCE, '--metric', METRIC, '--seed', str(SEED)]
subprocess.run(command + (['--runs', ONLY] if ONLY else []), cwd=REPO, env=env, check=True)

## 8. Figures

Per-subject accuracy of the runs, and each run's mean confusion matrix (every subject's matrix normalized by
the trials of each true class, then averaged over subjects) with its macro-F1 (mean over subjects).
Saved to `results/bcic2a/figures/`.

In [ ]:
import matplotlib.pyplot as plt

FIGURES = RESULTS / DATASET / 'figures'
FIGURES.mkdir(parents=True, exist_ok=True)
finished = {name: records for name, records in RECORDS.items() if records}

if finished:
    n_classes = len(next(iter(next(iter(finished.values())).values()))['class_names'])
    fig, ax = plt.subplots(figsize=(min(18, max(10, 0.3 * len(TARGETS) * max(len(finished), 2))), 3.8))
    width = 0.8 / len(finished)
    for k, (name, records) in enumerate(finished.items()):
        subjects = [s for s in TARGETS if s in records]
        xs = [TARGETS.index(s) + (k - (len(finished) - 1) / 2) * width for s in subjects]
        ax.bar(xs, [records[s]['test_acc'] * 100 for s in subjects], width, label=name)
    ax.axhline(100 / n_classes, color='gray', lw=0.8, ls='--', label='chance')
    ax.set(xticks=range(len(TARGETS)), ylabel='accuracy (%)', ylim=(0, 100), xlim=(-0.6, len(TARGETS) - 0.4))
    ax.set_xticklabels([f'S{s}' for s in TARGETS], fontsize=8 if len(TARGETS) <= 12 else 6, rotation=0 if len(TARGETS) <= 12 else 90)
    ax.set_title(f'{DATASET}, seed {SEED}', loc='left')
    ax.legend(fontsize=8, ncol=len(finished) + 1, loc='lower right', bbox_to_anchor=(1, 1), frameon=False)
    fig.tight_layout()
    fig.savefig(FIGURES / f'per_subject_accuracy_seed{SEED}.png', dpi=150)
    plt.show()

    fig, axes = plt.subplots(1, len(finished), figsize=(3.6 * len(finished), 3.5), squeeze=False)
    for ax, (name, records) in zip(axes[0], finished.items()):
        classes = next(iter(records.values()))['class_names']
        counts = np.array([r['confusion_counts'] for r in records.values()], dtype=float)
        recall = counts / counts.sum(axis=2, keepdims=True).clip(min=1)
        mean = recall.mean(axis=0) * 100
        precision = np.diagonal(counts, axis1=1, axis2=2) / counts.sum(axis=1).clip(min=1)
        diagonal = np.diagonal(recall, axis1=1, axis2=2)
        macro_f1 = np.mean(2 * precision * diagonal / (precision + diagonal).clip(min=1e-12), axis=1).mean()
        ax.imshow(mean, cmap='Blues', vmin=0, vmax=100)
        for i in range(len(classes)):
            for j in range(len(classes)):
                ax.text(j, i, f'{mean[i, j]:.1f}', ha='center', va='center', fontsize=8,
                        color='white' if mean[i, j] > 50 else 'black')
        labels = [c.replace('_', ' ') for c in classes]
        ax.set(xticks=range(len(classes)), yticks=range(len(classes)), xticklabels=labels, yticklabels=labels,
               xlabel='predicted', ylabel='true', title=f'{name}\nmacro-F1 {macro_f1 * 100:.2f}')
        ax.tick_params(axis='x', labelrotation=45, labelsize=8)
        ax.tick_params(axis='y', labelsize=8)
    fig.tight_layout()
    fig.savefig(FIGURES / f'mean_confusion_seed{SEED}.png', dpi=150)
    plt.show()
else:
    print('No finished folds yet.')

## 9. What the oscillatory branch learned

From `oscillatory_ssm` in every fold's record. Left: readout weight of the log band powers, summed per 1-Hz bin over
the states (all folds), when the run reads power. Right: readout weight of the synchrony features R_jk per probe and
band (mean over folds; probes 0 / 1 / 2 start as left / right / global with `hemi`). Below: per target subject (with
its accuracy), the three synchrony features and the three power states with the largest readout weight.
Saved to `results/<dataset>/figures/`.

In [ ]:
import matplotlib.pyplot as plt

osc_runs = {name: records for name, records in RECORDS.items()
            if any('oscillatory_ssm' in r for r in records.values())}
BINS = np.arange(0, 46)

for name, records in osc_runs.items():
    subjects = [s for s in TARGETS if s in records and 'oscillatory_ssm' in records[s]]
    infos = {s: records[s]['oscillatory_ssm'] for s in subjects}
    fig, axes = plt.subplots(1, 2, figsize=(14, 3.6))
    if all('readout_weight' in i for i in infos.values()):
        total = sum(np.histogram(np.ravel(i['frequency_hz']), bins=BINS, weights=np.ravel(i['readout_weight']))[0]
                    for i in infos.values())
        axes[0].bar(BINS[:-1] + 0.5, total / total.sum(), width=1.0, color='#4C72B0')
        axes[0].set(xlabel='centre frequency (Hz)', ylabel='share of readout weight', title=f'{name}: band power')
    else:
        axes[0].axis('off')
    if all('sync_readout_weight' in i for i in infos.values()):
        weight = np.mean([np.array(i['sync_readout_weight']) for i in infos.values()], axis=0)    # [J, K]
        freqs = np.mean([np.array(i['frequency_hz'])[0] for i in infos.values()], axis=0)
        image = axes[1].imshow(weight, aspect='auto', cmap='viridis')
        axes[1].set_xticks(range(len(freqs)))
        axes[1].set_xticklabels([f'{f:.1f}' for f in freqs], fontsize=8)
        axes[1].set_yticks(range(weight.shape[0]))
        axes[1].set(xlabel='band centre (Hz, mean over folds)', ylabel='probe', title='synchrony readout weight')
        fig.colorbar(image, ax=axes[1])
    else:
        axes[1].axis('off')
    fig.tight_layout()
    fig.savefig(FIGURES / f'oscillatory_readout_{name}_seed{SEED}.png', dpi=150)
    plt.show()

    print(name)
    for s in subjects:
        info, cells = infos[s], []
        freq = np.array(info['frequency_hz'])
        if 'sync_readout_weight' in info:
            w = np.array(info['sync_readout_weight'])
            for i in np.argsort(w.ravel())[::-1][:3]:
                j, k = divmod(int(i), w.shape[1])
                cells.append(f'R probe{j} {freq[0, k]:5.1f} Hz')
        if 'readout_weight' in info:
            w = np.array(info['readout_weight'])
            for i in np.argsort(w.ravel())[::-1][:3]:
                v, k = divmod(int(i), w.shape[1])
                cells.append(f'P v{v:02d} {freq[v, k]:5.1f} Hz')
        print(f'  S{s:<3d} acc {records[s]["test_acc"] * 100:6.2f}% | ' + ' | '.join(cells))
if not osc_runs:
    print('No finished fold of a run with the oscillatory branch yet.')

Per-fold confusion matrices and learning curves are in each run's `figures/`, the trained checkpoints in
`checkpoints/`, the resolved configuration in `config.yaml`. On Kaggle, save a version of the notebook to keep
`/kaggle/working`; attach that output to the next session to continue unfinished runs.